# Recommendation System

#### Definition:
A recommendation system is an intelligent algorithm designed to suggest items such as movies, products, music or services based on a userâ€™s past behavior, preferences or similarities with other users.

#### Business Problem:
Users face difficulty finding relevant items from a large selection, while businesses need to understand user preferences to provide personalized recommendations and improve engagement.

#### Project Objective:
The objective of this project is to develop a personalized recommendation system using usersâ€™ rating data. The system will analyze the relationship between users and items based on their ratings and use Item-Item Collaborative Filtering to generate relevant item recommendations for each user.



In [ ]:
#importing libraries

# Data handling
import numpy as np
import pandas as pd

# EDA
import matplotlib.pyplot as plt
import seaborn as sns

# Model Building
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

# Model Evaluation
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

# Deployment
import joblib


In [ ]:
# Loading Dataset
df = pd.read_csv("ratings.csv")
df.head()

In [ ]:
# rename the columns 
df.columns = ['UserID', 'ItemID', 'Rating', 'Timestamp']
df.head()

In [ ]:
# Information about the dataset
df.info()

In [ ]:
# Check for missing values
print("\nMissing values per column:")
display(df.isnull().sum())

In [ ]:
# Check for duplicate rows
print(f"\nNumber of duplicate rows: {df.duplicated().sum()}")

**Observation**:The dataset doesn't contain any missing values or duplicate values.

In [ ]:
# Statistical Summary for numerical columns
df.describe()

#### Observations:
- The Dataset contains of 7.82 million ratings .
- The average rating is 4.01, showing that ratings are generally positive.
- The median rating is 5, meaning at least half of the ratings are the maximum score.
- The standard deviation is 1.38, indicating noticeable variation in user ratings.
- 75% of ratings are 5 or below, and the 25th percentile is 3, showing that ratings are concentrated toward the higher end.

In [ ]:
# Statistical Summary for categorical columns
df.describe(include='object')

#### Observations:
- Dataset consiste of 4.20 million unique users.
- The dataset contains 476,001 unique products, indicating a large product catalog.
- The most active user has given 520 ratings.
- The most-rated product has received 18,244 ratings.

In [ ]:
print("Total Ratings:", len(df))
print("Unique Users:", df["UserID"].nunique())
print("Unique Products:", df["ItemID"].nunique())
print("Average Rating:", round(df["Rating"].mean(), 2))

In [ ]:
# Rating Count Distribution
rating_counts = df["Rating"].value_counts().sort_index()
print(rating_counts)
print("----------------------------------------------------------------")
plt.figure(figsize=(8, 5))
sns.countplot(
    data=df,
    x="Rating",
    order=sorted(df["Rating"].unique())
)
plt.title("Distribution of Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.show()

#### Observation:
The ratings are strongly skewed toward higher scores, with 5-star ratings dominating (4.35 million), followed by 4-star ratings (1.49 million). This indicates that users generally gave positive ratings to products.

In [ ]:
# Most Active Users
user_rating_counts = (
    df.groupby("UserID")
      .size()
      .sort_values(ascending=False)
)
print(user_rating_counts.head(10))

In [ ]:
# Top_users Distribution
top_users = user_rating_counts.head(10)
plt.figure(figsize=(10, 5))
sns.barplot(
    x=top_users.values,
    y=top_users.index
)
plt.title("Top 10 Most Active Users")
plt.xlabel("Number of Ratings")
plt.ylabel("User ID")
plt.show()

In [ ]:
# Most Rated Item
Item_rating_counts = (
    df.groupby("ItemID")
      .size()
      .sort_values(ascending=False)
)
print(Item_rating_counts.head(10))

In [ ]:
# Most rated items distribution
top_Items = Item_rating_counts.head(10)
plt.figure(figsize=(10, 6))
sns.barplot(
    x=top_Items.values,
    y=top_Items.index
)
plt.title("Top 10 Most Rated Items")
plt.xlabel("Number of Ratings")
plt.ylabel("Item ID")
plt.show()

In [ ]:
# Average Rating by Item 
Item_stats = df.groupby("ItemID").agg(
    Average_Rating=("Rating", "mean"),
    Rating_Count=("Rating", "count")
)

# Products having at least 50 ratings
popular_Items = Item_stats[
    Item_stats["Rating_Count"] >= 50
].sort_values(
    "Average_Rating",
    ascending=False
)
popular_Items.head(10)

In [ ]:
top_rated_Items = popular_Items.head(10)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_rated_Items,
    x="Average_Rating",
    y=top_rated_Items.index
)

plt.title("Top-Rated Items with Minimum 50 Ratings")
plt.xlabel("Average Rating")
plt.ylabel("Item ID")
plt.xlim(0, 5)
plt.show()

In [ ]:
# Rating vs Number of Ratings
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=Item_stats,
    x="Rating_Count",
    y="Average_Rating",
    alpha=0.5
)
plt.xscale("log")
plt.title("Average Rating vs Number of Ratings")
plt.xlabel("Number of Ratings")
plt.ylabel("Average Rating")
plt.ylim(0, 5)
plt.show()

### Insights from EDA:
- The dataset contains 7.82 million ratings and 4.20 million unique users providing a large volume of user-product interactions.
- The dataset contains 476,001 unique products, representing a large product catalog.
- The average rating is 4.01, indicating generally positive user feedback.
- The median rating is 5, showing a strong concentration of high ratings.
- 5-star ratings dominate with approximately 4.35 million ratings.
- 4-star ratings are the second most common with approximately 1.49 million ratings.
- The most active user has provided 520 ratings, showing variation in user activity.
- The most-rated product has received 18,244 ratings, indicating differences in product popularity.
- The dataset contains no missing or duplicate values, making it suitable for further modeling and analysis.

## Feature Engineering

In [ ]:
# ==============================
# USER RATING THRESHOLD
# ==============================

user_rating_counts = df["UserID"].value_counts()

thresholds = [1, 3, 5, 7, 10]
for t in thresholds:
    print(f"Users with more than {t} ratings:",
          (user_rating_counts > t).sum())

**Threshold Selection**:A minimum threshold of 5 ratings per user was selected to ensure sufficient user interaction history for reliable user profiling and clustering, while retaining a reasonable number of users (178,669).

In [ ]:
# User-features
user_features = df.groupby("UserID").agg(
    Average_Rating=("Rating", "mean"),
    Rating_Count=("Rating", "count"),
    Rating_Std=("Rating", "std"),
).reset_index()

user_features["Rating_Std"] = user_features["Rating_Std"].fillna(0)
user_features.head()

In [ ]:
# Apply user threshold
user_features = user_features[
    user_features["Rating_Count"] >= 5
].copy()

print("Users after filtering:", len(user_features))

In [ ]:
cluster_features = [
    "Average_Rating",
    "Rating_Count",
    "Rating_Std"
]    

X = user_features[cluster_features]
display(X.head())

In [ ]:
# Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Shape of scaled data:", X_scaled.shape)

## Clustering Models
we selected 3 clustering models:

1.Kmeans

2.Agglomerative

3.DBSCAN

### 1. K-Means Clustering

In [ ]:
#-------------------------
# Elbow Method
#-------------------------
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertia = []
k_values = range(2, 9)
for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    model.fit(X_scaled)
    inertia.append(model.inertia_)
    
    plt.figure(figsize=(8, 5))
plt.plot(
    k_values,
    inertia,
    marker="o"
)
plt.title("K-Means Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.show()

In [ ]:
#--------------------------
# Silhouette Scores
#-------------------------
from sklearn.metrics import silhouette_score

silhouette_scores = []
for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(
        X_scaled,
        labels,
        sample_size=min(50000, len(X_scaled)),
        random_state=42
    )
    silhouette_scores.append(score)
silhouette_df = pd.DataFrame({
    "K": list(k_values),
    "Silhouette_Score": silhouette_scores
})
display(silhouette_df)

In [ ]:
#Select Optimal K

optimal_k = int(
    silhouette_df.loc[
        silhouette_df["Silhouette_Score"].idxmax(),
        "K"]
)
print("Optimal number of clusters:", optimal_k)

In [ ]:
# Final K-Means

kmeans_model = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)
user_features["KMeans_Cluster"] = kmeans_model.fit_predict(X_scaled)
print("K-Means clustering completed.")

In [ ]:
#  K-Means Cluster Distribution

print(user_features["KMeans_Cluster"].value_counts().sort_index())

In [ ]:
user_features.groupby("KMeans_Cluster")[
    ["Average_Rating", "Rating_Count", "Rating_Std"]
].mean()

In [ ]:
sample_users = (
    user_features
    .groupby("KMeans_Cluster", group_keys=False)
    .apply(lambda x: x.sample(min(3, len(x)), random_state=42))
    [["KMeans_Cluster", "UserID", "Average_Rating", "Rating_Count"]]
)

display(sample_users)

## 2. Agglomerative Clustering

In [ ]:
# Create 10,000 User Sample

sample_size = min(10000, len(X_scaled))
rng = np.random.default_rng(42)
sample_indices = rng.choice(
    len(X_scaled),
    size=sample_size,
    replace=False
)
X_sample = X_scaled[sample_indices]
print("Sample size for Agglomerative:", X_sample.shape)

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

agg_model = AgglomerativeClustering(
    n_clusters=optimal_k,
    linkage="ward"
)

agg_labels = agg_model.fit_predict(X_sample)

print("Agglomerative clustering completed.")

In [ ]:
# Agglomerative Evaluation

agg_silhouette = silhouette_score(
    X_sample,
    agg_labels
)

agg_calinski = calinski_harabasz_score(
    X_sample,
    agg_labels
)

agg_davies = davies_bouldin_score(
    X_sample,
    agg_labels
)

print("Agglomerative Silhouette Score:", agg_silhouette)
print("Agglomerative Calinski-Harabasz Score:", agg_calinski)
print("Agglomerative Davies-Bouldin Score:", agg_davies)

## 3. DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

eps_values = [0.5, 0.7, 0.9, 1.1, 1.3, 1.5, 1.7, 2.0]

dbscan_results = []

for eps in eps_values:

    model = DBSCAN(
        eps=eps,
        min_samples=10
    )

    labels = model.fit_predict(X_sample)

    n_clusters = len(set(labels) - {-1})
    noise_count = np.sum(labels == -1)
    noise_percentage = (noise_count / len(labels)) * 100

    non_noise = labels != -1

    if n_clusters >= 2 and np.sum(non_noise) > n_clusters:

        sil = silhouette_score(
            X_sample[non_noise],
            labels[non_noise]
        )

        calinski = calinski_harabasz_score(
            X_sample[non_noise],
            labels[non_noise]
        )

        davies = davies_bouldin_score(
            X_sample[non_noise],
            labels[non_noise]
        )

    else:
        sil = np.nan
        calinski = np.nan
        davies = np.nan

    dbscan_results.append({
        "eps": eps,
        "Clusters": n_clusters,
        "Noise_Count": noise_count,
        "Noise_%": noise_percentage,
        "Silhouette": sil,
        "Calinski_Harabasz": calinski,
        "Davies_Bouldin": davies
    })

dbscan_tuning = pd.DataFrame(dbscan_results)

display(dbscan_tuning)

In [ ]:
# DBSCAN model fitting and evaluation

dbscan_model = DBSCAN(
    eps=0.7,
    min_samples=10
)

dbscan_labels = dbscan_model.fit_predict(X_sample)

non_noise = dbscan_labels != -1

dbscan_silhouette = silhouette_score(
    X_sample[non_noise],
    dbscan_labels[non_noise]
)

dbscan_calinski = calinski_harabasz_score(
    X_sample[non_noise],
    dbscan_labels[non_noise]
)

dbscan_davies = davies_bouldin_score(
    X_sample[non_noise],
    dbscan_labels[non_noise]
)

print("DBSCAN Results")
print("-------------------------")
print("Clusters:", len(set(dbscan_labels) - {-1}))
print("Noise points:", np.sum(dbscan_labels == -1))
print(
    "Noise percentage:",
    np.mean(dbscan_labels == -1) * 100
)
print("Silhouette Score:", dbscan_silhouette)
print("Calinski-Harabasz Score:", dbscan_calinski)
print("Davies-Bouldin Score:", dbscan_davies)

In [ ]:
#  K-Means Evaluation

kmeans_silhouette = silhouette_score(
    X_scaled,
    user_features["KMeans_Cluster"],
    sample_size=min(10000, len(X_scaled)),
    random_state=42
)

kmeans_calinski = calinski_harabasz_score(
    X_scaled,
    user_features["KMeans_Cluster"]
)

kmeans_davies = davies_bouldin_score(
    X_scaled,
    user_features["KMeans_Cluster"]
)

print("K-Means Silhouette:", kmeans_silhouette)
print("K-Means Calinski-Harabasz:", kmeans_calinski)
print("K-Means Davies-Bouldin:", kmeans_davies)

In [ ]:
#----------------------------------
# Clustering Comparison
#----------------------------------

clustering_comparison = pd.DataFrame({
    "Model": [
        "K-Means",
        "Agglomerative",
        "DBSCAN"
    ],

    "Silhouette Score": [
        kmeans_silhouette,
        agg_silhouette,
        dbscan_silhouette
    ],

    "Calinski-Harabasz Score": [
        kmeans_calinski,
        agg_calinski,
        dbscan_calinski
    ],

    "Davies-Bouldin Score": [
        kmeans_davies,
        agg_davies,
        dbscan_davies
    ]
})

display(clustering_comparison)

In [ ]:
# User Segment Summary

cluster_summary = user_features.groupby("KMeans_Cluster").agg(
    Average_Rating=("Average_Rating", "mean"),
    Average_Rating_Count=("Rating_Count", "mean"),
    Average_Rating_Std=("Rating_Std", "mean"),
    User_Count=("UserID", "count")
).reset_index()
display(cluster_summary)

In [ ]:
# Recommendation dataset Filtering

user_counts = df.groupby("UserID").size()
item_counts = df.groupby("ItemID").size()

eligible_users = user_counts[user_counts >= 5].index
eligible_items = item_counts[item_counts >= 20].index

recommendation_df = df[
    df["UserID"].isin(eligible_users) &
    df["ItemID"].isin(eligible_items)
].copy()

print("Recommendation data:", recommendation_df.shape)


In [ ]:
# User-level train-test split
def user_train_test_split(
    data,
    test_size=0.20,
    random_state=42
):
    
    rng = np.random.default_rng(random_state)

    train_parts = []
    test_parts = []

    for user_id, group in data.groupby("UserID"):

        # Shuffle the user's ratings
        indices = rng.permutation(len(group))

        # Number of ratings for testing
        n_test = max(
            1,
            int(np.ceil(len(group) * test_size))
        )

        # Keep at least one rating for training
        n_test = min(
            n_test,
            len(group) - 1
        )

        # 20% â†’ test
        test_idx = indices[:n_test]

        # 80% â†’ train
        train_idx = indices[n_test:]

        test_parts.append(
            group.iloc[test_idx]
        )

        train_parts.append(
            group.iloc[train_idx]
        )

    train_data = pd.concat(
        train_parts
    ).reset_index(drop=True)

    test_data = pd.concat(
        test_parts
    ).reset_index(drop=True)

    return train_data, test_data


In [ ]:
train, test = user_train_test_split(
    recommendation_df,
    test_size=0.20,
    random_state=42
)

print("Recommendation data:", recommendation_df.shape)
print("Training data:",train.shape)
print("Testing data:",test.shape)

In [ ]:
#-----------------------------
#Sparse Item-User Matrix
#----------------------------
import scipy.sparse as sp

items = pd.Index(train["ItemID"].unique())
users = pd.Index(train["UserID"].unique())

item_to_index = {
    item: i for i, item in enumerate(items)
}

user_to_index = {
    user: i for i, user in enumerate(users)
}

rows = train["ItemID"].map(item_to_index).values
columns = train["UserID"].map(user_to_index).values
values = train["Rating"].astype(np.float32).values

item_user_matrix = sp.csr_matrix(
    (values, (rows, columns)),
    shape=(len(items), len(users))
)

print("Sparse matrix shape:",item_user_matrix.shape)

In [ ]:
#User History
from collections import defaultdict
train_by_user = defaultdict(list)

for user_id, item_id, rating in train[
    ["UserID", "ItemID", "Rating"]
].itertuples(index=False):

    train_by_user[user_id].append(
        (item_id, rating)
    )
print("User history created.")

## Popularity Baseline

In [ ]:
popular_items = (
    train
    .groupby("ItemID")
    .size()
    .sort_values(ascending=False)
    .index
    .tolist()
)
print("Top 10 popular products:")
print(popular_items[:10])

In [ ]:
#Popularity Recommendation
def popularity_recommend(user_id,n=10):

    seen = {
        item_id
        for item_id, rating in train_by_user.get(user_id,[])
    }

    recommendations = [
        item_id
        for item_id in popular_items
        if item_id not in seen
    ]

    return recommendations[:n]

## Item-Item Collaborative Filtering

In [ ]:
knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=21,
    n_jobs=-1
)

knn_model.fit(item_user_matrix)

distances, neighbor_indices = knn_model.kneighbors(
        item_user_matrix,
        return_distance=True
    )

similarities = 1 - distances

print("Item-Item Collaborative Filtering model created.")

In [ ]:
# Recommendation Function

item_array = np.asarray(items)
item_index_dict = item_to_index


def item_cf_recommend(user_id, n=10):

    if user_id not in train_by_user:
        return popular_items[:n]

    seen = {
        item_id
        for item_id, rating in train_by_user[user_id]
    }

    scores = defaultdict(float)

    for item_id, rating in train_by_user[user_id]:

        if item_id not in item_index_dict:
            continue

        index = item_index_dict[item_id]

        for neighbor_index, similarity in zip(
            neighbor_indices[index, 1:],
            similarities[index, 1:]
        ):

            candidate = item_array[neighbor_index]

            if candidate in seen:
                continue

            scores[candidate] += (
                float(similarity) * float(rating)
            )

    ranked_items = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = [
        item_id for item_id, score in ranked_items
    ]

    # Fill remaining recommendations using popularity
    if len(recommendations) < n:

        for item_id in popular_items:

            if item_id not in seen and item_id not in recommendations:
                recommendations.append(item_id)

            if len(recommendations) >= n:
                break

    return recommendations[:n]

In [ ]:
#Test Recommendation
sample_user = test["UserID"].iloc[0]

recommendations = item_cf_recommend(
    sample_user,
    n=10
)

print("User ID:",sample_user)
print("Recommended Products:")
print(recommendations)

In [ ]:
#Prepare Evaluation Ground Truth

RATING_THRESHOLD = 4
test_items_by_user = (
    test[test["Rating"] >= RATING_THRESHOLD]
    .groupby("UserID")["ItemID"]
    .apply(set)
    .to_dict()
)

In [ ]:
# Compare Popularity and Item-CF

def popularity_recommend(user_id, n=10):
    seen = {
        item_id for item_id, rating
        in train_by_user.get(user_id, [])
    }
    return [
        item for item in popular_items
        if item not in seen
    ][:n]


def evaluate_model(recommendation_function, k=10):

    scores = []

    for user_id, actual_items in test_items_by_user.items():

        recommendations = recommendation_function(user_id, n=k)
        hits = set(recommendations) & actual_items
        h = len(hits)

        precision = h / k
        recall = h / len(actual_items)

        dcg = sum(
            1 / np.log2(rank + 1)
            for rank, item in enumerate(recommendations, 1)
            if item in actual_items
        )

        ideal = min(len(actual_items), k)
        idcg = sum(
            1 / np.log2(r + 1)
            for r in range(1, ideal + 1)
        )

        ndcg = dcg / idcg if idcg else 0

        scores.append([precision, recall, int(h > 0), ndcg])

    scores = np.mean(scores, axis=0)

    return scores


comparison = pd.DataFrame(
    [
        evaluate_model(popularity_recommend),
        evaluate_model(item_cf_recommend)
    ],
    index=["Popularity", "Item-CF"],
    columns=["Precision@10", "Recall@10", "Hit Rate@10", "NDCG@10"]
)

display(comparison)

#### Insights from Model Building:
- Three clustering models were implemented: **K-Means**, **Agglomerative Clustering**, and **DBSCAN**.
- DBSCAN achieved the strongest raw clustering scores on Silhouette and Davies-Bouldin, while **K-Means** provided a more practical approach for assigning new users.
- **Item-Item Collaborative Filtering** was implemented using **cosine similarity and KNN** to generate product recommendations.
- Models were evaluated using Silhouette, Calinski-Harabasz, Davies-Bouldin, and recommendation metrics such as Precision@10, Recall@10, Hit Rate@10, and NDCG@10.
- A popularity-based baseline was also created to compare recommendation performance.
- The overall modeling process successfully produced both user segments and personalized product recommendations, supporting the project's deployment objective.

# Deployment

In [ ]:
import joblib

artifacts = {
    "scaler": scaler,
    "kmeans_model": kmeans_model,
    "user_features": user_features,
    "item_user_matrix": item_user_matrix,
    "items": np.asarray(items),
    "item_to_index": item_to_index,
    "neighbor_indices": neighbor_indices,
    "similarities": similarities,
    "train_by_user": dict(train_by_user),
    "popular_items": popular_items
}

joblib.dump(artifacts, "recommendation_artifacts.pkl")

print("Model artifacts saved successfully.")

In [ ]:
import streamlit as st
import joblib
import pandas as pd
from collections import defaultdict, Counter


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Product Recommendation System",
    page_icon="ðŸ›ï¸",
    layout="wide"
)


# ============================================================
# MAIN TITLE
# ============================================================

st.title("ðŸ›ï¸ Product Recommendation System")

st.markdown(
    "### Item-Item Collaborative Filtering with User Clustering"
)


# ============================================================
# LOAD ARTIFACTS
# ============================================================

@st.cache_resource
def load_artifacts():

    artifacts = joblib.load(
        "recommendation_artifacts.pkl"
    )

    return artifacts


artifacts = load_artifacts()


# ============================================================
# EXTRACT ARTIFACTS
# ============================================================

user_features = artifacts["user_features"]

items = artifacts["items"]

item_to_index = artifacts["item_to_index"]

neighbor_indices = artifacts["neighbor_indices"]

similarities = artifacts["similarities"]

train_by_user = artifacts["train_by_user"]

popular_items = artifacts["popular_items"]


# ============================================================
# CLUSTER INFORMATION
# ============================================================

CLUSTER_NAMES = {
    0: "Highly Active Positive Users",
    1: "Highly Satisfied Users",
    2: "Lower Rated / Less Active Users"
}


# ============================================================
# SAMPLE USERS
# ============================================================

DEMO_USERS = {

    0: [
        "A2Y4HPQD4Z3KU",
        "A2WC9NFLG57W4H",
        "A3UKWQS8SRW6IO"
    ],

    1: [
        "AK9YBPCXTPBRW",
        "A2F6SZLKG0TOJ7",
        "A1Q671DHPH8LME"
    ],

    2: [
        "A2I2WS8UL72QFY",
        "A1Q2W9BBY3I5J2",
        "ACT7FYXKJPAJ5"
    ]
}


# ============================================================
# SAMPLE ITEMS
# ============================================================

SAMPLE_ITEMS = list(items[:10])


# ============================================================
# ITEM STATISTICS
# ============================================================

@st.cache_data
def create_item_statistics():

    item_ratings = defaultdict(list)

    for user_id, ratings in train_by_user.items():

        for item_id, rating in ratings:

            item_ratings[item_id].append(
                float(rating)
            )

    rows = []

    for item_id, ratings in item_ratings.items():

        rows.append(
            {
                "Item ID": item_id,

                "Average Rating": round(
                    sum(ratings) / len(ratings),
                    2
                ),

                "Rating Count": len(ratings)
            }
        )

    return pd.DataFrame(rows)


item_stats = create_item_statistics()


# ============================================================
# GET ITEM INFORMATION
# ============================================================

def get_item_information(item_id):

    if item_stats.empty:

        return {
            "exists": False,
            "avg_rating": None,
            "rating_count": 0,
            "cluster": None
        }


    matching_rows = item_stats[
        item_stats["Item ID"] == item_id
    ]


    if matching_rows.empty:

        return {
            "exists": False,
            "avg_rating": None,
            "rating_count": 0,
            "cluster": None
        }


    row = matching_rows.iloc[0]


    # --------------------------------------------------------
    # FIND USERS WHO RATED THIS ITEM
    # --------------------------------------------------------

    item_users = []


    for user_id, ratings in train_by_user.items():

        for rated_item, rating in ratings:

            if rated_item == item_id:

                item_users.append(user_id)

                break


    # --------------------------------------------------------
    # FIND USER CLUSTERS
    # --------------------------------------------------------

    clusters = []


    for user_id in item_users:

        user_rows = user_features[
            user_features["UserID"] == user_id
        ]


        if not user_rows.empty:

            cluster = int(
                user_rows.iloc[0]["KMeans_Cluster"]
            )

            clusters.append(cluster)


    # --------------------------------------------------------
    # DOMINANT CLUSTER
    # --------------------------------------------------------

    if clusters:

        dominant_cluster = Counter(
            clusters
        ).most_common(1)[0][0]

    else:

        dominant_cluster = None


    return {
        "exists": True,

        "avg_rating": row["Average Rating"],

        "rating_count": int(
            row["Rating Count"]
        ),

        "cluster": dominant_cluster
    }


# ============================================================
# GET USER INFORMATION
# ============================================================

def get_user_information(user_id):

    rows = user_features[
        user_features["UserID"] == user_id
    ]


    if rows.empty:

        return None


    row = rows.iloc[0]


    return {
        "cluster": int(
            row["KMeans_Cluster"]
        ),

        "average_rating": float(
            row["Average_Rating"]
        ),

        "rating_count": int(
            row["Rating_Count"]
        ),

        "rating_std": float(
            row["Rating_Std"]
        )
    }


# ============================================================
# USER RECOMMENDATION FUNCTION
# ============================================================

def recommend_products(
    user_id,
    top_n
):

    # --------------------------------------------------------
    # NEW USER / COLD START
    # --------------------------------------------------------

    if user_id not in train_by_user:

        return list(
            popular_items[:top_n]
        )


    user_history = train_by_user[user_id]


    if not user_history:

        return list(
            popular_items[:top_n]
        )


    # --------------------------------------------------------
    # ITEM-ITEM COLLABORATIVE FILTERING
    # --------------------------------------------------------

    scores = defaultdict(float)

    rated_items = set()


    for item_id, rating in user_history:

        rated_items.add(item_id)


        if item_id not in item_to_index:

            continue


        item_index = item_to_index[item_id]


        neighbors = neighbor_indices[
            item_index
        ]


        neighbor_similarities = similarities[
            item_index
        ]


        for neighbor_index, similarity in zip(
            neighbors,
            neighbor_similarities
        ):

            recommended_item = items[
                neighbor_index
            ]


            # Don't recommend already-rated items

            if recommended_item in rated_items:

                continue


            scores[recommended_item] += (
                float(similarity)
                * float(rating)
            )


    # --------------------------------------------------------
    # SORT RECOMMENDATIONS
    # --------------------------------------------------------

    ranked_items = sorted(
        scores.items(),

        key=lambda x: x[1],

        reverse=True
    )


    recommendations = [

        item_id

        for item_id, score

        in ranked_items[:top_n]

    ]


    # --------------------------------------------------------
    # FALLBACK
    # --------------------------------------------------------

    if not recommendations:

        recommendations = list(
            popular_items[:top_n]
        )


    return recommendations


# ============================================================
# ITEM SIMILARITY RECOMMENDATION
# ============================================================

def recommend_similar_items(
    item_id,
    top_n
):

    # --------------------------------------------------------
    # NEW ITEM
    # --------------------------------------------------------

    if item_id not in item_to_index:

        return list(
            popular_items[:top_n]
        )


    item_index = item_to_index[
        item_id
    ]


    neighbors = neighbor_indices[
        item_index
    ]


    neighbor_similarities = similarities[
        item_index
    ]


    results = []


    for neighbor_index, similarity in zip(
        neighbors,
        neighbor_similarities
    ):

        recommended_item = items[
            neighbor_index
        ]


        if recommended_item == item_id:

            continue


        results.append(
            (
                recommended_item,
                float(similarity)
            )
        )


    results = sorted(
        results,

        key=lambda x: x[1],

        reverse=True
    )


    return [

        item_id

        for item_id, similarity

        in results[:top_n]

    ]


# ============================================================
# CREATE RECOMMENDATION TABLE
# ============================================================

def create_recommendation_table(
    recommendations
):

    rows = []


    for rank, item_id in enumerate(
        recommendations,
        start=1
    ):

        information = get_item_information(
            item_id
        )


        cluster = information[
            "cluster"
        ]


        rows.append(
            {
                "Rank": rank,

                "Item ID": item_id,

                # Only show cluster number
                # 0, 1 or 2

                "Cluster": (
                    cluster
                    if cluster is not None
                    else "N/A"
                ),

                "Average Rating": (
                    information["avg_rating"]
                    if information["avg_rating"]
                    is not None
                    else "N/A"
                )
            }
        )


    return pd.DataFrame(rows)


# ============================================================
# USER RATING HISTORY
# ============================================================

def get_user_rating_history(
    user_id
):

    if user_id not in train_by_user:

        return pd.DataFrame()


    history = train_by_user[
        user_id
    ]


    return pd.DataFrame(
        history,

        columns=[
            "Item ID",
            "Rating"
        ]
    )


# ============================================================
# ITEM RATING HISTORY
# ============================================================

def get_item_rating_history(
    item_id
):

    rows = []


    for user_id, ratings in train_by_user.items():

        for rated_item, rating in ratings:

            if rated_item == item_id:

                rows.append(
                    {
                        "User ID": user_id,
                        "Rating": rating
                    }
                )


    return pd.DataFrame(rows)


# ============================================================
# USER PROFILE BOX
# ============================================================

def display_user_profile(
    user_id,
    user_info
):

    # --------------------------------------------------------
    # NEW USER
    # --------------------------------------------------------

    if user_info is None:

        st.info(
            f"""
            ### ðŸ†• New User

            **User ID:** `{user_id}`

            **Status:** No historical information available

            **Recommendation Type:** Popular products
            (Cold Start)
            """
        )

        return


    # --------------------------------------------------------
    # GET CLUSTER
    # --------------------------------------------------------

    cluster = user_info[
        "cluster"
    ]


    cluster_name = CLUSTER_NAMES.get(
        cluster,
        "Unknown Cluster"
    )


    # --------------------------------------------------------
    # CLUSTER 0
    # BLUE INFO BOX
    # --------------------------------------------------------

    if cluster == 0:

        st.info(
            f"""
            ### ðŸ”µ Cluster 0

            {cluster_name}

            **User ID:** `{user_id}`
            """
        )


    # --------------------------------------------------------
    # CLUSTER 1
    # GREEN SUCCESS BOX
    # --------------------------------------------------------

    elif cluster == 1:

        st.success(
            f"""
            ### ðŸŸ¢ Cluster 1

            {cluster_name}

            **User ID:** `{user_id}`
            """
        )


    # --------------------------------------------------------
    # CLUSTER 2
    # RED ERROR BOX
    # --------------------------------------------------------

    elif cluster == 2:

        st.error(
            f"""
            ### ðŸ”´ Cluster 2

            {cluster_name}

            **User ID:** `{user_id}`
            """
        )


    # --------------------------------------------------------
    # UNKNOWN CLUSTER
    # --------------------------------------------------------

    else:

        st.warning(
            f"""
            ### Cluster {cluster}

            {cluster_name}

            **User ID:** `{user_id}`
            """
        )


# ============================================================
# SIDEBAR NAVIGATION
# ============================================================

st.sidebar.title(
    "ðŸ“Œ Navigation"
)


page = st.sidebar.radio(
    "Select Recommendation Type",

    [
        "ðŸ‘¤ User ID Recommendations",
        "ðŸ›ï¸ Item ID Recommendations"
    ]
)


# ============================================================
# NUMBER OF RECOMMENDATIONS SLIDER
# ============================================================

st.sidebar.markdown(
    "---"
)


st.sidebar.subheader(
    "Recommendation Settings"
)


top_n = st.sidebar.slider(
    "Number of Recommendations",

    min_value=1,

    max_value=20,

    value=10,

    step=1
)


st.sidebar.caption(
    f"Showing {top_n} recommendation(s)"
)


# ============================================================
# USER ID RECOMMENDATION PAGE
# ============================================================

if page == "ðŸ‘¤ User ID Recommendations":

    # --------------------------------------------------------
    # PAGE TITLE
    # --------------------------------------------------------

    st.markdown(
    '<p style="font-size:24px; font-weight:600; margin-top:10px; margin-bottom:15px;">'
    ' User ID Recommendation System :'
    '</p>',
    unsafe_allow_html=True
    )


    # --------------------------------------------------------
    # USER SELECTION
    # --------------------------------------------------------

    st.subheader(
        "User Selection"
    )


    selection_type = st.radio(
        "Select User Type",

        [
            "Sample User",
            "Enter User ID"
        ],

        horizontal=True
    )


    # --------------------------------------------------------
    # SAMPLE USER
    # --------------------------------------------------------

    if selection_type == "Sample User":

        selected_cluster = st.selectbox(
            "Select Sample Cluster",

            options=[
                0,
                1,
                2
            ],

            format_func=lambda x:
                f"Cluster {x} - "
                f"{CLUSTER_NAMES[x]}"
        )


        user_id = st.selectbox(
            "Select Sample User ID",

            DEMO_USERS[
                selected_cluster
            ]
        )


    # --------------------------------------------------------
    # MANUAL USER ID
    # --------------------------------------------------------

    else:

        user_id = st.text_input(
            "Enter User ID"
        ).strip()


    # --------------------------------------------------------
    # GENERATE RECOMMENDATIONS
    # --------------------------------------------------------

    if st.button(
        "Generate Recommendations",
        type="primary"
    ):

        if not user_id:

            st.warning(
                "Please enter or select a User ID."
            )


        else:

            # ------------------------------------------------
            # GENERATE RECOMMENDATIONS
            # ------------------------------------------------

            recommendations = recommend_products(
                user_id,
                top_n
            )


            # =================================================
            # USER PROFILE
            # =================================================

            st.subheader(
                "User Profile"
            )


            user_info = get_user_information(
                user_id
            )


            # ------------------------------------------------
            # COLORED USER PROFILE BOX
            # ------------------------------------------------

            display_user_profile(
                user_id,
                user_info
            )


            # =================================================
            # USER METRICS
            # =================================================

            if user_info is not None:

                col1, col2, col3 = st.columns(
                    3
                )


                col1.metric(
                    "Average Rating",

                    round(
                        user_info[
                            "average_rating"
                        ],
                        2
                    )
                )


                col2.metric(
                    "Rating Count",

                    user_info[
                        "rating_count"
                    ]
                )


                col3.metric(
                    "Rating Std",

                    round(
                        user_info[
                            "rating_std"
                        ],
                        2
                    )
                )


            # =================================================
            # RECOMMENDED PRODUCTS
            # =================================================

            st.subheader(
                "Recommended Products"
            )


            recommendation_df = (
                create_recommendation_table(
                    recommendations
                )
            )


            st.dataframe(
                recommendation_df,

                use_container_width=True,

                hide_index=True
            )


            # =================================================
            # USER RATING HISTORY
            # =================================================

            st.subheader(
                "User Rating History"
            )


            history_df = (
                get_user_rating_history(
                    user_id
                )
            )


            if not history_df.empty:

                st.dataframe(
                    history_df,

                    use_container_width=True,

                    hide_index=True
                )

            else:

                st.info(
                    "No rating history available "
                    "for this user."
                )


# ============================================================
# ITEM ID RECOMMENDATION PAGE
# ============================================================

else:

    # --------------------------------------------------------
    # PAGE TITLE
    # --------------------------------------------------------

    st.markdown(
    '<p style="font-size:24px; font-weight:600; margin-top:10px; margin-bottom:15px;">'
    ' Item ID Recommendation System :'
    '</p>',
    unsafe_allow_html=True
    )


    # --------------------------------------------------------
    # ITEM SELECTION
    # --------------------------------------------------------

    st.subheader(
        "Item Selection"
    )


    selection_type = st.radio(
        "Select Item Type",

        [
            "Sample Item",
            "Enter Item ID"
        ],

        horizontal=True
    )


    # --------------------------------------------------------
    # SAMPLE ITEM
    # --------------------------------------------------------

    if selection_type == "Sample Item":

        item_id = st.selectbox(
            "Select Sample Item ID",

            SAMPLE_ITEMS
        )


    # --------------------------------------------------------
    # MANUAL ITEM ID
    # --------------------------------------------------------

    else:

        item_id = st.text_input(
            "Enter Item ID"
        ).strip()


    # --------------------------------------------------------
    # FIND SIMILAR PRODUCTS
    # --------------------------------------------------------

    if st.button(
        "Find Similar Products",
        type="primary"
    ):

        if not item_id:

            st.warning(
                "Please enter or select an Item ID."
            )


        else:

            item_information = (
                get_item_information(
                    item_id
                )
            )


            # =================================================
            # EXISTING ITEM
            # =================================================

            if item_information[
                "exists"
            ]:

                cluster = item_information[
                    "cluster"
                ]


                # ------------------------------------------------
                # SELECTED ITEM
                # ------------------------------------------------

                st.subheader(
                    "Selected Item"
                )


                if cluster == 0:

                    st.info(
                        f"""
                        ### ðŸ”µ Cluster 0

                        **Item ID:** `{item_id}`

                        {CLUSTER_NAMES[0]}
                        """
                    )


                elif cluster == 1:

                    st.success(
                        f"""
                        ### ðŸŸ¢ Cluster 1

                        **Item ID:** `{item_id}`

                        {CLUSTER_NAMES[1]}
                        """
                    )


                elif cluster == 2:

                    st.error(
                        f"""
                        ### ðŸ”´ Cluster 2

                        **Item ID:** `{item_id}`

                        {CLUSTER_NAMES[2]}
                        """
                    )


                else:

                    st.warning(
                        f"""
                        ### Cluster N/A

                        **Item ID:** `{item_id}`

                        Unknown
                        """
                    )


                # ------------------------------------------------
                # ITEM METRICS
                # ------------------------------------------------

                col1, col2, col3 = st.columns(
                    3
                )


                col1.metric(
                    "Average Rating",

                    item_information[
                        "avg_rating"
                    ]
                )


                col2.metric(
                    "Rating Count",

                    item_information[
                        "rating_count"
                    ]
                )


                col3.metric(
                    "Cluster",

                    cluster
                    if cluster is not None
                    else "N/A"
                )


                # ------------------------------------------------
                # SIMILAR ITEMS
                # ------------------------------------------------

                recommendations = (
                    recommend_similar_items(
                        item_id,
                        top_n
                    )
                )


                # =================================================
                # RECOMMENDED PRODUCTS
                # =================================================

                st.subheader(
                    "Recommended Products"
                )


                recommendation_df = (
                    create_recommendation_table(
                        recommendations
                    )
                )


                st.dataframe(
                    recommendation_df,

                    use_container_width=True,

                    hide_index=True
                )


                # =================================================
                # ITEM RATING HISTORY
                # =================================================

                st.subheader(
                    "Item Rating History"
                )


                item_history = (
                    get_item_rating_history(
                        item_id
                    )
                )


                if not item_history.empty:

                    st.dataframe(
                        item_history,

                        use_container_width=True,

                        hide_index=True
                    )

                else:

                    st.info(
                        "No historical ratings available "
                        "for this item."
                    )


            # =================================================
            # NEW ITEM
            # =================================================

            else:

                st.subheader(
                    "Selected Item"
                )


                st.warning(
                    f"""
                    ### ðŸ†• New Item

                    **Item ID:** `{item_id}`

                    **Status:** No historical information available
                    """
                )


                # ------------------------------------------------
                # NEW ITEM METRICS
                # ------------------------------------------------

                col1, col2, col3 = st.columns(
                    3
                )


                col1.metric(
                    "Status",
                    "New Item"
                )


                col2.metric(
                    "Rating Count",
                    0
                )


                col3.metric(
                    "Cluster",
                    "N/A"
                )


                st.info(
                    "This item is not available in the "
                    "recommendation model. Popular products "
                    "are shown as cold-start recommendations."
                )


                # =================================================
                # POPULAR PRODUCTS
                # =================================================

                st.subheader(
                    "Popular Products"
                )


                recommendations = list(
                    popular_items[:top_n]
                )


                recommendation_df = (
                    create_recommendation_table(
                        recommendations
                    )
                )


                st.dataframe(
                    recommendation_df,

                    use_container_width=True,

                    hide_index=True
                )


                # =================================================
                # ITEM RATING HISTORY
                # =================================================

                st.subheader(
                    "Item Rating History"
                )


                st.info(
                    "No historical information available "
                    "for this item."
                )